#  Import libraries

In [ ]:
!pip install requirements.txt

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# 2. Data Acquisation

In [4]:
# To store temporarly scarped data
data = []

for page in range(1, 51):
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    for book in soup.find_all("article", class_="product_pod"):
        title = book.find('h3').find('a')['title']
        price = book.find('p', class_='price_color').text.strip()
        rating = book.find('p', class_='star-rating')['class'][1]
        stock = book.find('p', class_='instock availability').text.strip()
        book_link = book.find('h3').find('a')['href']
        book_url = 'https://books.toscrape.com/catalogue/' + book_link.replace('../', '')

        # Visit each page
        book_response = requests.get(book_url)

        book_soup = BeautifulSoup(
            book_response.text,
            'html.parser'
        )

        breadcrumb = book_soup.find('ul', class_='breadcrumb')

        category = breadcrumb.find_all('li')[2].text.strip()

        # Store scraped data
        data.append({
            'Title': title,
            'Price': price,
            'Rating': rating,
            'Stock': stock,
            'Category': category
        })

    print(f"Page {page} completed")

    # Small delay between pages
    time.sleep(0.2)


Page 1 completed
Page 2 completed
Page 3 completed
Page 4 completed
Page 5 completed
Page 6 completed
Page 7 completed
Page 8 completed
Page 9 completed
Page 10 completed
Page 11 completed
Page 12 completed
Page 13 completed
Page 14 completed
Page 15 completed
Page 16 completed
Page 17 completed
Page 18 completed
Page 19 completed
Page 20 completed
Page 21 completed
Page 22 completed
Page 23 completed
Page 24 completed
Page 25 completed
Page 26 completed
Page 27 completed
Page 28 completed
Page 29 completed
Page 30 completed
Page 31 completed
Page 32 completed
Page 33 completed
Page 34 completed
Page 35 completed
Page 36 completed
Page 37 completed
Page 38 completed
Page 39 completed
Page 40 completed
Page 41 completed
Page 42 completed
Page 43 completed
Page 44 completed
Page 45 completed
Page 46 completed
Page 47 completed
Page 48 completed
Page 49 completed
Page 50 completed


In [5]:
# Store collected data to a dataframe
books_df = pd.DataFrame(data)

print(f"Total books:{len(books_df)}")
books_df.head()

Total books:1000


,Title,Price,Rating,Stock,Category
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction
2,Soumission,Â£50.10,One,In stock,Fiction
3,Sharp Objects,Â£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History


In [6]:
books_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Title     1000 non-null   str  
 1   Price     1000 non-null   str  
 2   Rating    1000 non-null   str  
 3   Stock     1000 non-null   str  
 4   Category  1000 non-null   str  
dtypes: str(5)
memory usage: 39.2 KB


In [8]:
books_df.shape

(1000, 5)

In [11]:
books_df.isna().sum()

Title       0
Price       0
Rating      0
Stock       0
Category    0
dtype: int64

In [13]:
books_df.dtypes

Title       str
Price       str
Rating      str
Stock       str
Category    str
dtype: object

# 2. Data Cleaning
After eximining the dataset, we have no misssing values. 
We will only clean price, category, and create a numerical(bool) value for stock

In [16]:
# Clean price
books_df['Price'] = (books_df['Price'].astype(str).str.replace(r'[^0-9]', '', regex=True).astype(float))

# Clean category
books_df['Category'] = (books_df['Category'].astype(str).str.strip())

# Clean stock
books_df['Stock'] = (books_df['Stock'].astype(str).str.replace('\n', '', regex=False).str.strip())

# Create numerical boolean value for stock
books_df['In_Stock'] = books_df['Stock'].str.contains('In stock', case=False, na=False).astype(int)

clean_df = books_df.copy()
clean_df.info()
clean_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Title     1000 non-null   str    
 1   Price     1000 non-null   float64
 2   Rating    1000 non-null   str    
 3   Stock     1000 non-null   str    
 4   Category  1000 non-null   str    
 5   In_Stock  1000 non-null   int64  
dtypes: float64(1), int64(1), str(4)
memory usage: 47.0 KB


,Title,Price,Rating,Stock,Category,In_Stock
0,A Light in the Attic,517700.0,Three,In stock,Poetry,1
1,Tipping the Velvet,537400.0,One,In stock,Historical Fiction,1
2,Soumission,501000.0,One,In stock,Fiction,1
3,Sharp Objects,478200.0,Four,In stock,Mystery,1
4,Sapiens: A Brief History of Humankind,542300.0,Five,In stock,History,1
